# SC3000 DPO — Original instructor Step 1–8 Notebook

Original instructional code and all eight headings are retained. Only the four course-completion steps are supplemented with the historical Random E5 training chain and its data/checkpoint integrity checks. Data-generation source is kept in corresponding `scripts/*.py` rather than duplicated as twelve giant cells. CUDA execution and saved outputs still need to be verified on TC1.

### Step 1: Install necesscary packages

In [ ]:
# Instructor's original installation commands, optional with preconfigured TC1 .venv:
# !pip install matplotlib
# !pip install torch numpy transformers datasets tiktoken wandb tqdm
import torch
print("Using preinstalled torch",torch.__version__)


### Step 2: Package imports and configuration

In [ ]:
import sys
import os
from pathlib import Path

# Run this notebook either from the repository root or the dpo/ folder.
# Never depend on a particular user's home-directory location.
work_dir = Path.cwd().resolve()
if (work_dir / "model.py").is_file() and (work_dir / "dpo" / "dpo.ipynb").is_file():
    repo_root = work_dir
elif (work_dir / "dpo.ipynb").is_file() and (work_dir.parent / "model.py").is_file():
    repo_root = work_dir.parent
else:
    raise RuntimeError("Open the notebook from the NanoGPT-Math root or dpo/ directory.")
dpo_dir = repo_root / "dpo"
sys.path.insert(0, str(repo_root))
# Keep Slurm-assigned CUDA_VISIBLE_DEVICES; do not force GPU 1.
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
# Keep 'course' as the historical baseline. Choose 'answer_focused' for Round 1D.
loss_mode = 'course'  # 'course' or 'answer_focused'
answer_weight = 1.0   # chosen numeric-answer negative log-likelihood
completion_weight = 0.25  # chosen full-response NLL to reinforce the output template
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with (repo_root / "sft" / "meta.pkl").open("rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])
print(f"Repository root: {repo_root}")
print(f"Selected device: {device}")

# Historical E5 continuation configuration (original settings above unchanged).
FAST_SMOKE=False
FINAL_EXPORT_POLICY="archived_if_available"
AUTO_BUILD_MISSING_DATA=True
from collections import Counter,defaultdict
from datetime import datetime,timezone
import hashlib,shutil,subprocess,re
sys.path.insert(0,str(repo_root/"scripts"))
SFT=repo_root/"sft/gpt.pt"
META=repo_root/"sft/meta.pkl"
RUN_DIR=repo_root/"dpo/experiments"/("final_notebook_smoke" if FAST_SMOKE else "final_notebook_e5_reproduction")
RUN_DIR.mkdir(parents=True,exist_ok=True)
MAX_NEW_TOKENS=80
BETA=beta
SEED=42
BATCH_SIZE=64
TRAIN_BASE_LENGTH=64
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for part in iter(lambda:f.read(1048576),b""):h.update(part)
    return h.hexdigest()
def read_json(p):return json.loads(Path(p).read_text(encoding="utf-8"))
def save_json_once(path,val):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():raise FileExistsError(path)
    path.write_text(json.dumps(val,indent=2,ensure_ascii=False)+"\n",encoding="utf-8")
env_record={"python":sys.version,"torch":torch.__version__,
            "sft_sha256":sha256(SFT),"tokenizer_meta_sha256":sha256(META),
            "device":device,"cuda":torch.version.cuda}
if not FAST_SMOKE and device!="cuda":raise RuntimeError("Full historical training requires CUDA")


### Step 3: Define helper functions

In [3]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

def answer_focused_masks(input_ids):
    """Return masks over next-character targets for the response and number.

    Every preference row has format:
       <math prompt ending with '=?'> The answer is <integer>.
    Question/prompt token probabilities are deliberately excluded.
    Integer answers match the original model.py period-based stopping rule.
    """
    if input_ids.ndim != 2 or input_ids.size(1) < 3:
        raise ValueError("Expected [batch, seq] token IDs")
    if '=' not in stoi or '?' not in stoi:
        raise ValueError("Tokenizer must support the math question marker =?")
    marker = ((input_ids[:, :-1] == stoi['=']) &
              (input_ids[:, 1:] == stoi['?']))
    if not bool(marker.any(dim=1).all().item()):
        raise ValueError("Every preference example must contain =?")
    B, L = input_ids.shape
    # marker[j] means the '?' is character j+1 in the full text.
    absolute_positions = torch.arange(1, L, device=input_ids.device)
    qmark = torch.where(marker, absolute_positions.unsqueeze(0), -1).max(dim=1).values
    valid = input_ids[:, 1:] != 0  # newline ID 0 also serves as padding
    response_mask = (absolute_positions.unsqueeze(0) > qmark[:, None]) & valid
    # The chosen/rejected response is required to start with this shared template.
    template = " The answer is "
    template_ids = torch.tensor(encode(template), dtype=torch.long,
                                device=input_ids.device)
    for i in range(B):
        start = int(qmark[i].item()) + 1
        if start + len(template) > L or not torch.equal(
            input_ids[i, start:start + len(template)], template_ids
        ):
            raise ValueError("Round 1D requires the response template ' The answer is '")
    number_start = qmark + 1 + len(template)
    # Exclude the final period from the numerical NLL: prioritize the sign/digits.
    number_mask = ((absolute_positions.unsqueeze(0) >= number_start[:, None]) &
                   valid & (input_ids[:, 1:] != stoi['.']))
    if not bool(number_mask.any(dim=1).all().item()):
        raise ValueError("Every preference response needs a numeric answer")
    return response_mask, number_mask


def compute_answer_focused_logprob(input_ids):
    """Teacher-forced scores for the response and its numeric answer only.

    Returns (response_logprob_sum, response_logprob_mean, number_logprob_mean).
    The preference term uses SUM, not mean. Positive NLL focuses on numeric
    answer tokens, with a smaller template/completion supervision term.
    """
    logits, _ = gpt(input_ids[:, :-1], full_seq=True)
    targets = input_ids[:, 1:]
    token_logprobs = F.log_softmax(logits, dim=-1).gather(
        -1, targets.unsqueeze(-1)
    ).squeeze(-1)
    response_mask, number_mask = answer_focused_masks(input_ids)
    response_sum = (token_logprobs * response_mask).sum(dim=1)
    response_mean = response_sum / response_mask.sum(dim=1).clamp_min(1)
    number_sum = (token_logprobs * number_mask).sum(dim=1)
    number_mean = number_sum / number_mask.sum(dim=1).clamp_min(1)
    return response_sum, response_mean, number_mean


### Step 4: Load the pretrained NanoGPT model

In [ ]:
ckpt = torch.load(repo_root / "sft" / "gpt.pt", map_location=device, weights_only=True)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()


### Step 5: Load Data (**students are required to complete this part!**)

In [ ]:
# Step 5: load and validate preference pairs.
# The supplied pos_neg_pairs.json has only four DEMO examples.
# Replace/extend it with a larger, checked preference dataset for full training.
data_path = dpo_dir / "pos_neg_pairs.json"
if not data_path.exists():
    data_path = dpo_dir / "data" / "pos_neg_pairs.json"
if not data_path.exists():
    raise FileNotFoundError("Missing dpo/pos_neg_pairs.json")

with data_path.open("r", encoding="utf-8") as f:
    raw_pairs = json.load(f)
if not isinstance(raw_pairs, list) or not raw_pairs:
    raise ValueError("Expected a nonempty JSON array of positive/negative pairs")

lines = []
for index, pair in enumerate(raw_pairs):
    if not isinstance(pair, dict) or not all(k in pair for k in ("positive", "negative")):
        raise ValueError(f"Sample {index} must contain positive and negative strings")
    pos, neg = pair["positive"], pair["negative"]
    if not isinstance(pos, str) or not isinstance(neg, str) or pos == neg:
        raise ValueError(f"Invalid responses at sample {index}")

    # The question ends with '=?' in the course's arithmetic/algebra format.
    # This is more reliable than finding the longest common prefix: two numeric
    # answers can share initial digits, which would mask away answer tokens.
    end = pos.find("=?")
    if end < 0:
        raise ValueError(f"Sample {index}: missing '=?' in positive response")
    prompt = pos[:end + 2]
    if not neg.startswith(prompt):
        raise ValueError(f"Sample {index}: positive/negative prompts do not match")
    for response in (pos, neg):
        missing = sorted(set(response) - set(stoi))
        if missing:
            raise ValueError(f"Sample {index} contains unknown characters: {missing}")
    lines.append({"positive": pos, "negative": neg, "prompt": prompt})

max_pair_length = max(max(len(pair["positive"]), len(pair["negative"])) for pair in lines)
# The course get_batches() appends four newline padding markers before packing.
required_length = max_pair_length + 4
if required_length > gpt.config.block_size:
    raise ValueError(
        f"Longest training sequence ({required_length}) exceeds "
        f"model block_size ({gpt.config.block_size})"
    )
max_length = max(max_length, required_length)  # Do not truncate the question.
if len(lines) < batch_size:
    print(f"Only {len(lines)} pairs: shrinking batch_size for a pipeline smoke test.")
    batch_size = len(lines)
print(f"Loaded {len(lines)} preference pairs from {data_path}")
print(f"batch_size={batch_size}, max_length={max_length}")
if len(lines) <= 4:
    print("WARNING: These are the four course examples, NOT a real training dataset.")


**Data generation for the selected E5 (Step 5 extension).** Only the pipeline dependencies, data-generation entrypoints, source SHA256 and frozen split checks are shown below. Full `.py` generator code remains in the repository and should be included in the submission package; unrelated experiments are not embedded.

In [ ]:
# Step 5 extension: only E5-required data generators and immutable source manifests.
# Generator implementations are kept as separate .py files for submission.
GENERATOR_PLAN=[
("round1","dpo/data/round1_full/round1_manifest.json","scripts/generate_round1.py",["--profile","full"]),
("round1b","dpo/experiments/round1b_integer/round1b_manifest.json","scripts/generate_round1b.py",[]),
("round1c","dpo/experiments/round1c_incorrect/round1c_manifest.json","scripts/generate_round1c.py",[]),
("round3","dpo/data/round3_structured_full/manifest.json","scripts/generate_round3_structured.py",["--profile","full"]),
("round4","dpo/data/round4_matched_full/manifest.json","scripts/build_round4_data.py",["--profile","full"]),
("round5","dpo/data/round5_decomposition/manifest.json","scripts/build_round5_decomposition.py",[]),
("round7","dpo/data/round7_targeted_full/manifest.json","scripts/build_round7_targeted_data.py",["--profile","full"])]
generator_provenance={}
for label,manifest,script,args in GENERATOR_PLAN:
    file=repo_root/script
    if not file.is_file():raise FileNotFoundError(file)
    generator_provenance[script]=sha256(file)
    target=repo_root/manifest
    if not target.exists():
        if not AUTO_BUILD_MISSING_DATA:raise FileNotFoundError(target)
        subprocess.run([sys.executable,str(file),*args],cwd=repo_root,check=True)
    generator_provenance[manifest]=sha256(target)
    print("DATA VERIFIED",label)
TEACHER_DIRS=[(repo_root/"dpo/benchmarks/teacher_style_400_v1",400),(repo_root/"dpo/benchmarks/teacher_style_extension_280_v1",280)]
for folder,count in TEACHER_DIRS:
    m=read_json(folder/"manifest.json")
    assert sha256(folder/"questions.json")==m["questions_sha256"]
    assert len(read_json(folder/"questions.json"))==count


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [ ]:
# Step 6: build the optimizer and learning-rate scheduler.
# No separate SFT training or reference model is needed for the course loss.
import math

optimizer = torch.optim.AdamW(
    gpt.parameters(),
    lr=base_lr,
    weight_decay=0.01,
)

# The course-provided get_batches() skips the final incomplete batch.
steps_per_epoch = len(lines) // batch_size
if steps_per_epoch == 0:
    raise ValueError("No full batch available: reduce batch_size or add preference data.")
if len(lines) % batch_size:
    print(
        f"Note: {len(lines) % batch_size} example(s) per epoch will be skipped "
        "by the instructor's get_batches() helper."
    )
total_steps = epochs * steps_per_epoch
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=base_lr * 0.1
)
print(f"AdamW lr={base_lr}, epochs={epochs}, steps_per_epoch={steps_per_epoch}")


### Step 7: Begin training (**students are required to complete this part!**)
    
**Two reproducible loss modes:** `course` preserves the instructor's original
whole-sequence mean objective. `answer_focused` uses completion-only
preference sums plus extra correct-number and completion NLL terms. The latter
is a controlled additional experiment and is not standard reference-model DPO.


In [ ]:
# Teacher original Step 7 code below, unchanged inside optional single-stage demo.
# Do NOT execute it during E5 reproduction, as this would overwrite dpo.pt.
RUN_SINGLE_STAGE_DEMO=False
if RUN_SINGLE_STAGE_DEMO:
    # Step 7: course baseline OR optional Round 1D answer-focused preference loss.
    # The original course branch below is preserved for reproducibility.
    # The new branch is a documented loss experiment, not standard reference-model DPO.
    random.seed(42)
    torch.manual_seed(42)
    gpt.train()
    training_loss = []

    if loss_mode not in ('course', 'answer_focused'):
        raise ValueError("loss_mode must be 'course' or 'answer_focused'")

    for epoch in range(epochs):
        running_loss = 0.0
        completed_batches = 0
        pbar = tqdm(
            get_batches(lines, batch_size),
            total=steps_per_epoch,
            desc=f"Epoch {epoch + 1}/{epochs}",
        )

        for neg_tensor, pos_tensor in pbar:
            optimizer.zero_grad(set_to_none=True)
            if loss_mode == 'course':
                # Instructor's unmodified whole-sequence mean preference loss.
                neg_logprob = compute_logprob(neg_tensor)
                pos_logprob = compute_logprob(pos_tensor)
                preference_loss = -F.logsigmoid(
                    (pos_logprob - neg_logprob) / beta
                ).mean()
                chosen_bonus = -0.1 * pos_logprob.mean()
            else:
                # Only completion tokens contribute to preference comparisons.
                pos_sum, pos_completion_mean, pos_number_mean = (
                    compute_answer_focused_logprob(pos_tensor)
                )
                neg_sum, _, _ = compute_answer_focused_logprob(neg_tensor)
                preference_loss = -F.logsigmoid(
                    (pos_sum - neg_sum) / beta
                ).mean()
                # Chosen token supervision: strong on digits, lighter on template.
                chosen_bonus = (
                    -answer_weight * pos_number_mean.mean()
                    -completion_weight * pos_completion_mean.mean()
                )
            loss = preference_loss + chosen_bonus

            if not torch.isfinite(loss).item():
                raise RuntimeError(f"Non-finite loss at epoch {epoch + 1}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gpt.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            completed_batches += 1
            running_loss += loss.item()
            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                pref=f"{preference_loss.item():.4f}",
                chosen=f"{chosen_bonus.item():.4f}",
            )

        if completed_batches == 0:
            raise RuntimeError("Training did not process any batch")
        mean_loss = running_loss / completed_batches
        training_loss.append(mean_loss)
        print(f"Epoch {epoch + 1}/{epochs}: mean loss={mean_loss:.4f} mode={loss_mode}")

        ckpt_path = dpo_dir / "dpo.pt"
        torch.save({
            "model_state_dict": gpt.state_dict(),
            "model_args": ckpt["model_args"],
        }, ckpt_path)
        print(f"Saved checkpoint to {ckpt_path}")

    print("Training completed.")
    print("Important: loss alone is not arithmetic/algebra accuracy.")

else:
    print("Teacher single-stage training retained; historical E5 training follows.")


**E5 addition to Step 7.** The instructor's original training cell above is retained as an optional demonstration. The selected E5 uses the same Course Loss and training semantics, but needs the historically correct multi-stage data, new AdamW at each stage, cosine horizons, and per-epoch checkpoints. The following implementation calls only the necessary external data generators—not twelve embedded copies of historical scripts.

In [ ]:
# Step 3 — Original teacher-forced token log-probabilities and batch helpers
from generate_round1 import parse_relation, stable_split

def stage_encode(s):
    return [stoi[ch] for ch in s]

def stage_pad_or_truncate(chars, max_length):
    return chars[-max_length:] if len(chars)>max_length else chars+[0]*(max_length-len(chars))

def stage_compute_logprob(gpt, batch_ids):
    inputs = batch_ids[:, :-1]
    targets = batch_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    loss = F.cross_entropy(logits.reshape(-1,logits.size(-1)),
                           targets.reshape(-1), ignore_index=0,
                           reduction="none").reshape_as(targets)
    mask = (targets!=0).float()
    return -(loss*mask).sum(dim=1)/mask.sum(dim=1)

def stage_course_preference_loss(gpt, negatives, positives):
    neg_lp = stage_compute_logprob(gpt, negatives)
    pos_lp = stage_compute_logprob(gpt, positives)
    return -F.logsigmoid((pos_lp-neg_lp)/BETA).mean()-0.1*pos_lp.mean()

def stage_get_batches(lines, batch_size, max_length, target_device):
    # EXACT instructor rule: shuffle rows in place, discard trailing <batch.
    random.shuffle(lines)
    for i in range(0,len(lines),batch_size):
        batch=lines[i:i+batch_size]
        if len(batch)<batch_size:continue
        neg=[stage_pad_or_truncate(stage_encode(x["negative"]+"\n\n\n\n"),max_length) for x in batch]
        pos=[stage_pad_or_truncate(stage_encode(x["positive"]+"\n\n\n\n"),max_length) for x in batch]
        yield (torch.tensor(neg,dtype=torch.long,device=target_device),
               torch.tensor(pos,dtype=torch.long,device=target_device))

def verify_pairs(path, expected_count=100000, smoke=False):
    source = read_json(path)
    if len(source)!=expected_count:
        raise RuntimeError(f"{path}: expected {expected_count}, found {len(source)}")
    rows = source[:128] if smoke else source
    max_required = TRAIN_BASE_LENGTH
    unique_prompts=set()
    unique_facts=set()
    for i,row in enumerate(rows):
        pos,neg=row["positive"],row["negative"]
        marker=" The answer is "
        if marker not in pos or " because " not in pos:
            raise ValueError(f"{path.name} row {i}: incorrect chosen format")
        question,answer_suffix=pos.split(marker,1)
        answer=answer_suffix.split(" because ",1)[0]
        if not re.fullmatch(r"[+-]?\d+",answer) or not pos.endswith("."):
            raise ValueError(f"{path.name} row {i}: malformed integer chosen")
        if neg!=question+" Sorry, I don't know.":
            raise ValueError(f"{path.name} row {i}: changed rejection/prompt")
        if set(pos+neg)-set(stoi):
            raise ValueError(f"{path.name} row {i}: tokenizer OOV")
        max_required=max(max_required,len(pos)+4,len(neg)+4)
        unique_prompts.add(question)
        unique_facts.add(parse_relation(question,answer))
    # Original NanoGPT block size=256; 64 was only the initial pack length.
    if max_required>256:
        raise ValueError(f"{path.name}: {max_required} exceeds model block_size")
    info={"source":str(path.relative_to(repo_root)),"sha256":sha256(path),
          "rows":len(rows),"unique_prompts":len(unique_prompts),
          "unique_facts":len(unique_facts),"max_length":max_required}
    return rows,info

def model_from_checkpoint(path):
    obj=torch.load(path,map_location="cpu",weights_only=True)
    args=obj["model_args"]
    model=GPT(GPTConfig(**args))
    state=obj.get("model_state_dict",obj.get("model"))
    if state is None:raise RuntimeError(f"{path}: missing state dict")
    model.load_state_dict({k.removeprefix("_orig_mod."):v for k,v in state.items()},strict=True)
    return model,args

# Sanity checks: answer-first comparison, no generated completion needed in training.
assert stage_encode("3*17=?")
assert abs(float(F.logsigmoid(torch.tensor(0.0)))+math.log(2))<1e-6
print("PASS: course loss, tokenizer, model loading and batch conventions",flush=True)

In [ ]:
# Step 6 — Historical optimizer and scheduler hyperparameters
R3 = repo_root / "dpo/data/round3_structured_full"
R4 = repo_root / "dpo/data/round4_matched_full"
R5 = repo_root / "dpo/data/round5_decomposition"
R7 = repo_root / "dpo/data/round7_targeted_full"
R9 = repo_root / "dpo/data/round9_balanced_full"

STAGE1_TO_4=[
    {"id":"round3_structured", "data":R3/"pos_neg_pairs.json",
     "manifest":R3/"manifest.json","hash_section":"sha256","hash_key":"train",
     "lr":1e-4,"actual_epochs":5,"schedule_epochs":5},
    {"id":"round4_treatment_e1","data":R4/"treatment_pairs.json",
     "manifest":R4/"manifest.json","hash_section":"file_sha256","hash_key":"treatment",
     "lr":2e-5,"actual_epochs":1,"schedule_epochs":2},
    {"id":"round5_original","data":R5/"original_pairs.json",
     "manifest":R5/"manifest.json","hash_section":"file_sha256","hash_key":"original",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
    {"id":"round7_replay","data":R7/"replay_pairs.json",
     "manifest":R7/"manifest.json","hash_section":"file_sha256","hash_key":"replay",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
]
EXPECTED_RANDOM_SHA="30519fe6883f175e67791a0461afe23e9212f38d974778c4bd5f552d3e26e3fa"

# Basic contract checks (also repeated in the resumable stage runner).
assert [x["id"] for x in STAGE1_TO_4]==[
    "round3_structured","round4_treatment_e1","round5_original","round7_replay"]
assert [x["actual_epochs"] for x in STAGE1_TO_4]==[5,1,1,1]
assert [x["schedule_epochs"] for x in STAGE1_TO_4]==[5,2,1,1]
print("PASS: 4 historical stages and their original cosine schedules")

In [ ]:
# Step 6 — Instructor's original AdamW + cosine scheduler, made reusable
# because the chosen E5 was trained in SEVEN sequential stages.
def build_optimizer_scheduler(gpt,base_lr,steps_per_epoch,schedule_epochs):
    optimizer = torch.optim.AdamW(gpt.parameters(),lr=base_lr,weight_decay=0.01)
    total_steps=schedule_epochs*steps_per_epoch
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,T_max=total_steps,eta_min=base_lr*0.1)
    return optimizer,scheduler

print("Step 6 PASS: original AdamW, cosine annealing, fresh restart at each stage.")

In [ ]:
# Step 7 — Course Loss training loop with epoch-boundary resume.
def atomic_save_torch(target, obj):
    target=Path(target)
    if target.exists():
        raise FileExistsError(f"Refusing to overwrite checkpoint: {target}")
    tmp=target.with_suffix(target.suffix+".partial")
    if tmp.exists():
        raise RuntimeError(f"Incomplete temporary checkpoint: {tmp}. Inspect first.")
    torch.save(obj, tmp)
    tmp.rename(target)

def stage_paths(stage_id):
    folder=RUN_DIR/stage_id
    return folder,folder/"progress.json"

def stage_finished_checkpoint(spec):
    folder,_=stage_paths(spec["id"])
    n=1 if FAST_SMOKE else spec["actual_epochs"]
    return folder/f"epoch_{n:02d}.pt"

def train_stage(spec, parent_path):
    assert Path(parent_path).is_file(),f"Missing parent checkpoint {parent_path}"
    data_file=Path(spec["data"])
    source_m=read_json(spec["manifest"])
    expected=source_m[spec["hash_section"]][spec["hash_key"]]
    if sha256(data_file)!=expected:
        raise RuntimeError(f"Data SHA256 does not match frozen source manifest: {data_file}")
    allrows,info=verify_pairs(data_file,smoke=FAST_SMOKE)
    batch=16 if FAST_SMOKE else BATCH_SIZE
    actual=1 if FAST_SMOKE else spec["actual_epochs"]
    schedule_epochs=spec["schedule_epochs"]
    steps=len(allrows)//batch
    if steps<=0:raise RuntimeError("Not enough preferences for one minibatch")
    protocol={
        "stage":spec["id"],"parent_sha256":sha256(parent_path),
        "dataset_sha256":info["sha256"],
        "source_manifest_sha256":sha256(spec["manifest"]),
        "seed":SEED,"batch_size":batch,"steps_per_epoch":steps,
        "actual_epochs":actual,"cosine_schedule_epochs":schedule_epochs,
        "learning_rate":spec["lr"],"loss":"instructor full-sequence mean",
        "max_length":info["max_length"],"beta":BETA,
        "model_block_size":256,"fast_smoke":FAST_SMOKE
    }
    stage_dir,progress_path=stage_paths(spec["id"])
    if progress_path.is_file():
        progress=read_json(progress_path)
        if progress["protocol"]!=protocol:
            raise RuntimeError("Refusing incompatible resume: stage data/schedule/parent changed")
        finished=progress["completed_epochs"]
        if not 1<=finished<=actual:
            raise RuntimeError("Invalid saved epoch offset")
        epoch_file=stage_dir/f"epoch_{finished:02d}.pt"
        if sha256(epoch_file)!=progress["last_checkpoint_sha256"]:
            raise RuntimeError(f"Corrupt stage checkpoint {epoch_file}")
        if finished==actual:
            print(f"SKIP verified {spec['id']}: {finished}/{actual} complete",flush=True)
            return epoch_file
        print(f"RESUME {spec['id']} from epoch {finished}/{actual}",flush=True)
    elif stage_dir.exists():
        # A failed allocation before the first optimizer update may have
        # written protocol.json but no checkpoint yet. Restart that *empty*
        # stage safely without overwriting any saved model.
        existing={p.name for p in stage_dir.iterdir()}
        if existing=={"protocol.json"} and read_json(stage_dir/"protocol.json")==protocol:
            finished=0
            progress=None
            print(f"RESTART EMPTY STAGE {spec['id']}: no completed epoch lost",flush=True)
        else:
            raise RuntimeError(f"Stale/incomplete stage folder: {stage_dir}; "
                               "inspect or rename; never overwrite saved epochs")
    else:
        stage_dir.mkdir(parents=True,exist_ok=False)
        save_json_once(stage_dir/"protocol.json",protocol)
        finished=0
        progress=None

    # Previous Python/CUDA seed reset BEFORE constructing GPT, just like all
    # historical train_round*.py stage entry points.
    random.seed(SEED)
    torch.manual_seed(SEED)
    if device=="cuda":torch.cuda.manual_seed_all(SEED)
    gpt, model_args=model_from_checkpoint(parent_path)
    if model_args["vocab_size"]!=len(stoi) or model_args["block_size"]!=256:
        raise RuntimeError("Original character tokenizer / model architecture changed")
    gpt.to(device).train()
    opt,scheduler=build_optimizer_scheduler(gpt,spec["lr"],steps,schedule_epochs)
    lines=allrows

    if finished:
        ck=torch.load(epoch_file,map_location="cpu",weights_only=True)
        if ck["stage_protocol"]!=protocol or ck["epoch"]!=finished:
            raise RuntimeError("Resume checkpoint metadata mismatch")
        gpt.load_state_dict(ck["model_state_dict"],strict=True)
        opt.load_state_dict(ck["optimizer_state_dict"])
        scheduler.load_state_dict(ck["scheduler_state_dict"])
        lines=ck["shuffled_lines"]
        random.setstate(ck["python_rng_state"])
        torch.set_rng_state(ck["torch_rng_state"])
        if device=="cuda":
            torch.cuda.set_rng_state_all(ck["cuda_rng_states"])
        if len(lines)!=len(allrows):
            raise RuntimeError("Saved shuffle order lost rows")

    for epoch in range(finished+1,actual+1):
        gpt.train()
        accumulated=0.
        updated=0
        for step,(neg,pos) in enumerate(stage_get_batches(lines,batch,info["max_length"],device),1):
            opt.zero_grad(set_to_none=True)
            loss=stage_course_preference_loss(gpt,neg,pos)
            if not bool(torch.isfinite(loss).item()):
                raise RuntimeError(f"Non-finite loss: {spec['id']} epoch={epoch} step={step}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gpt.parameters(),max_norm=1.0)
            opt.step()
            scheduler.step()
            updated+=1
            accumulated+=float(loss.item())
            if step==1 or step%250==0 or step==steps:
                print(f"TRAIN {spec['id']} epoch={epoch}/{actual} "
                      f"step={step}/{steps} loss={loss.item():.5f} "
                      f"lr={scheduler.get_last_lr()[0]:.9g}",flush=True)
        if updated!=steps:raise RuntimeError("Optimizer step count mismatch")
        ckpt=stage_dir/f"epoch_{epoch:02d}.pt"
        obj={
            "epoch":epoch,"stage_protocol":protocol,
            "model_state_dict":gpt.state_dict(),"model_args":model_args,
            "optimizer_state_dict":opt.state_dict(),
            "scheduler_state_dict":scheduler.state_dict(),
            "shuffled_lines":lines,"python_rng_state":random.getstate(),
            "torch_rng_state":torch.get_rng_state(),
            "cuda_rng_states":torch.cuda.get_rng_state_all() if device=="cuda" else []
        }
        atomic_save_torch(ckpt,obj)
        history=progress["history"] if progress else []
        history.append({"epoch":epoch,"mean_loss":accumulated/steps,
                        "checkpoint":ckpt.name,"sha256":sha256(ckpt)})
        progress={"protocol":protocol,"completed_epochs":epoch,
                  "last_checkpoint_sha256":sha256(ckpt),"history":history}
        temp=progress_path.with_suffix(".json.partial")
        if temp.exists():raise RuntimeError(f"Stale progress marker: {temp}")
        temp.write_text(json.dumps(progress,indent=2)+"\n",encoding="utf-8")
        temp.replace(progress_path)
        print(f"EPOCH SAVED {spec['id']} {epoch}/{actual} "
              f"mean_loss={accumulated/steps:.6f}",flush=True)
    print(f"STAGE PASS {spec['id']} -> {ckpt.name}",flush=True)
    return ckpt

# Verify the original SFT config without changing any model weights.
print("Step 7 training engine PASS: loss, optimizer, checkpoint and RNG resume ready",flush=True)

In [ ]:
# Step 7A — Train/reuse Round3, Round4 E1, Round5 Original, Round7 Replay
stage_outputs={"sft":SFT}
parent=SFT
for spec in STAGE1_TO_4:
    parent=train_stage(spec,parent)
    stage_outputs[spec["id"]]=parent
    print(f"CHAIN {spec['id']}: SHA256 {sha256(parent)[:20]}",flush=True)

ROUND7_PARENT=stage_outputs["round7_replay"]
print(f"PASS: Round7 Replay predecessor created/verified: {ROUND7_PARENT}",flush=True)

In [ ]:
# Step 7B — Verify/build the Round9 Random 100k training dataset
EXISTING_R9_DATA=R9/"random_pairs.json"
if EXISTING_R9_DATA.exists():
    r9_manifest=read_json(R9/"manifest.json")
    if r9_manifest["version"]!="round9_structured_v1":
        raise RuntimeError("Wrong historical Round9 manifest")
    if sha256(EXISTING_R9_DATA)!=r9_manifest["files_sha256"]["random"]:
        raise RuntimeError("Historical Random100k data SHA256 mismatch")
    random_data_path=EXISTING_R9_DATA
    random_manifest_path=R9/"manifest.json"
else:
    if not AUTO_BUILD_MISSING_DATA:
        raise FileNotFoundError("Missing Round9 Random preference dataset")
    import build_round9_data as b9
    import evaluate_teacher_style_680 as teacher680_module
    # Strictly validate both existing frozen teacher benchmarks.
    teacher680_module.validate()
    generated=RUN_DIR/"notebook_generated_round9"
    if generated.exists():
        if not (generated/"random_pairs.json").exists():
            raise RuntimeError("Incomplete generated Round9 folder; inspect it")
        random_data_path=generated/"random_pairs.json"
        random_manifest_path=generated/"manifest.json"
    else:
        # The original builder expects the parent at a historical absolute
        # path. Use our new Round7 checkpoint, without copying it there.
        historical_base=b9.BASE
        try:
            b9.BASE=ROUND7_PARENT
            r9_pairs,r9_metadata,r9_manifest=b9.make()
        finally:
            b9.BASE=historical_base
        generated.mkdir(parents=True,exist_ok=False)
        for arm in ("replay","random","structured"):
            (generated/f"{arm}_pairs.json").write_bytes(b9.bytes_json(r9_pairs[arm]))
        (generated/"structured_metadata.json").write_bytes(
            b9.bytes_json(r9_metadata))
        (generated/"manifest.json").write_bytes(b9.bytes_json(r9_manifest))
        random_data_path=generated/"random_pairs.json"
        random_manifest_path=generated/"manifest.json"

r9_manifest=read_json(random_manifest_path)
if sha256(random_data_path)!=r9_manifest["files_sha256"]["random"]:
    raise RuntimeError("Generated Random source mismatch with its own manifest")
if sha256(random_data_path)!=EXPECTED_RANDOM_SHA:
    raise RuntimeError(
        "Round9 Random data do NOT match archived E5 training source. "
        "Do not publish an alleged historical reproduction with changed data. "
        f"observed={sha256(random_data_path)} expected={EXPECTED_RANDOM_SHA}")
print(f"PASS: historical Random100k exact pair SHA={sha256(random_data_path)}")
print(f"Dataset location: {random_data_path}",flush=True)

In [ ]:
# Step 7C — Continue original staged training until selected E5
STAGE5_TO_7=[
    {"id":"round9_random_e1", "data":random_data_path,
     "manifest":random_manifest_path,"hash_section":"files_sha256","hash_key":"random",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
    {"id":"round9_random_e2_restart", "data":random_data_path,
     "manifest":random_manifest_path,"hash_section":"files_sha256","hash_key":"random",
     "lr":1e-5,"actual_epochs":1,"schedule_epochs":1},
    {"id":"round9_random_e3_e5_from_e2", "data":random_data_path,
     "manifest":random_manifest_path,"hash_section":"files_sha256","hash_key":"random",
     "lr":1e-5,"actual_epochs":3,"schedule_epochs":4}
]
assert [s["actual_epochs"] for s in STAGE5_TO_7]==[1,1,3]
assert [s["schedule_epochs"] for s in STAGE5_TO_7]==[1,1,4]

parent=ROUND7_PARENT
for spec in STAGE5_TO_7:
    parent=train_stage(spec,parent)
    stage_outputs[spec["id"]]=parent
    print(f"CHAIN {spec['id']} SHA256={sha256(parent)[:20]}",flush=True)

REPRODUCED_E5=stage_outputs["round9_random_e3_e5_from_e2"]
print(f"FINAL E5 REPLAY CHECKPOINT = {REPRODUCED_E5}",flush=True)

### Step 8: Begin testing (**students are required to complete this part!**)

**Additional E5 evaluation and provenance.** This is supplementary to, not a replacement for, the teacher's original Step 8 test questions above. The selected model is evaluated on development splits and its checkpoint lineage audited.

In [ ]:
# Step 8 — Original greedy generation and answer-first evaluation
ANSWER_RE=re.compile(r"^\s*The answer is\s+([+-]?\d+)(?=\s|[.]|$)",re.I)
EXPLANATION_RE=re.compile(
    r"^\s*The answer is\s+([+-]?\d+)\s+because\s+"
    r"([+-]?\d+)([+\-*/])([+-]?\d+)\s+equals\s+([+-]?\d+)\.\s*$",re.I)

def greedy_original_stop(model,prompt,limit=80):
    tokens=encode(prompt)
    if len(tokens)>model.config.block_size:
        raise RuntimeError("Evaluation prompt exceeds NanoGPT context")
    produced=[]
    stop="max_new_tokens"
    with torch.inference_mode():
        for _ in range(limit):
            x=torch.tensor([tokens[-model.config.block_size:]],
                           dtype=torch.long,device=device)
            logits,_=model(x)
            new_id=int(logits[0,-1].argmax().item())
            if new_id==0:
                stop="newline";break
            tokens.append(new_id)
            produced.append(new_id)
            if new_id==7:
                stop="period";break
    return "".join(itos[t] for t in produced),stop

def grade_completion(response,gold):
    m=ANSWER_RE.match(response)
    if m is None:return "format_failure",None,False
    got=int(m.group(1))
    status="correct" if got==int(gold) else "math_error"
    proof=EXPLANATION_RE.fullmatch(response)
    valid=False
    if proof:
        a,op,b,rhs=int(proof[2]),proof[3],int(proof[4]),int(proof[5])
        if op=="+":value=a+b
        elif op=="-":value=a-b
        elif op=="*":value=a*b
        else:value=(a/b if b else None)
        valid=(value==rhs==got)
    return status,got,valid

def evaluate_frozen(model,records,split):
    output=[]
    for i,item in enumerate(records):
        gold=str(item["answer"])
        response,stop=greedy_original_stop(model,item["prompt"],MAX_NEW_TOKENS)
        status,predicted,proof=grade_completion(response,gold)
        output.append({"prompt":item["prompt"],"answer":gold,
                       "category":item["category"],"completion":response,
                       "status":status,"prediction":predicted,
                       "stop_reason":stop,"explanation_valid":proof})
        if i<5:
            print(f"{split}: {item['prompt']} gold={gold} "
                  f"model={response!r} [{status}]",flush=True)
    n=len(output)
    count=sum(x["status"]=="correct" for x in output)
    by_cat={}
    for category in sorted({x["category"] for x in output}):
        rows=[x for x in output if x["category"]==category]
        by_cat[category]={"correct":sum(x["status"]=="correct" for x in rows),
                          "n":len(rows)}
    metrics={"correct":count,"n":n,"accuracy":count/n,"by_category":by_cat,
             "format_failure":sum(x["status"]=="format_failure" for x in output),
             "valid_explanations":sum(x["explanation_valid"] for x in output)}
    print(f"EVAL {split}: {count}/{n} ({count/n:.2%}) "
          f"format_failure={metrics['format_failure']}",flush=True)
    return output,metrics

r3_manifest=read_json(R3/"manifest.json")
for name,path,tag in [
    ("val",R3/"val.json","val"),
    ("train_probe",R3/"train_probe_256.json","train_probe")
]:
    if sha256(path)!=r3_manifest["sha256"][tag]:
        raise RuntimeError(f"Frozen Round3 {name} checksum mismatch")

teacher_core=read_json(TEACHER_DIRS[0][0]/"questions.json")
teacher_extension=read_json(TEACHER_DIRS[1][0]/"questions.json")
from generate_round1 import RESERVED
reserved_prompts=[
    "17+19=?","3*17=?","72/4=?","72-x=34,x=?","x*11=44,x=?"]
reserved_records=[
    {"prompt":p,"answer":RESERVED[p],
     "category":("multiply" if p=="3*17=?" else
                 "add" if p=="17+19=?" else
                 "divide" if p=="72/4=?" else
                 "a_minus_x" if p=="72-x=34,x=?" else "x_times_a")}
    for p in reserved_prompts]

dev_splits={
    "val":read_json(R3/"val.json"),
    "train_probe":read_json(R3/"train_probe_256.json"),
    "teacher680":teacher_core+teacher_extension,
    "reserved5":reserved_records
}
if FAST_SMOKE:
    dev_splits={name:rows[:12] for name,rows in dev_splits.items()}
assert (FAST_SMOKE or {k:len(v) for k,v in dev_splits.items()}==
        {"val":768,"train_probe":256,"teacher680":680,"reserved5":5})

evaluation_dir=RUN_DIR/"development_e5"
evaluation_dir.mkdir(parents=True,exist_ok=True)
model,_=model_from_checkpoint(REPRODUCED_E5)
model.to(device).eval()
measured={}
for split,rows in dev_splits.items():
    pred_file=evaluation_dir/f"{split}_predictions.json"
    stat_file=evaluation_dir/f"{split}_metrics.json"
    if pred_file.exists() and stat_file.exists():
        pred=read_json(pred_file)
        metric=read_json(stat_file)
        if len(pred)!=len(rows) or metric["correct"]!=sum(x["status"]=="correct" for x in pred):
            raise RuntimeError(f"Saved {split} score was corrupted")
        print(f"REUSE FROZEN EVAL {split}: {metric['correct']}/{metric['n']}")
    elif pred_file.exists() or stat_file.exists():
        raise RuntimeError(f"Only part of frozen {split} evaluation exists")
    else:
        pred,metric=evaluate_frozen(model,rows,split)
        save_json_once(pred_file,pred)
        save_json_once(stat_file,metric)
    measured[split]=metric
del model
if device=="cuda":torch.cuda.empty_cache()

# Historic values from TC1 saved runs; NOT the new run's measured values.
historical_e5={"val":614,"train_probe":226,"teacher680":521,"reserved5":4}
print("ACTUAL fresh-run scores:")
for k,v in measured.items():
    print(f"  {k:13} {v['correct']:4}/{v['n']:<4} "
          f"historical E5 reference: {historical_e5[k]}",flush=True)
# Evaluate the actual exported checkpoint as well when it is the archived
# historical E5, rather than assuming fresh re-training produced the exact
# same weights. These are the same frozen development splits, not final Test.
archived_for_eval=repo_root/"dpo/experiments/round9_random_multi_e3_e6/epoch_e05.pt"
submission_eval_source=(archived_for_eval
                        if FINAL_EXPORT_POLICY=="archived_if_available" and
                           archived_for_eval.is_file()
                        else REPRODUCED_E5)
measured_submission=measured
if submission_eval_source!=REPRODUCED_E5 and not FAST_SMOKE:
    print("EVALUATING SELECTED HISTORICAL E5 for submitted-model scores",flush=True)
    submission_eval_dir=RUN_DIR/"submitted_e5_development"
    submission_eval_dir.mkdir(parents=True,exist_ok=True)
    selected_model,_=model_from_checkpoint(submission_eval_source)
    selected_model.to(device).eval()
    measured_submission={}
    for split,rows in dev_splits.items():
        pred_file=submission_eval_dir/f"{split}_predictions.json"
        stat_file=submission_eval_dir/f"{split}_metrics.json"
        if pred_file.exists() and stat_file.exists():
            pred=read_json(pred_file)
            metric=read_json(stat_file)
            if len(pred)!=len(rows) or metric["correct"]!=sum(
                    x["status"]=="correct" for x in pred):
                raise RuntimeError(f"Corrupt submitted E5 eval: {split}")
        elif pred_file.exists() or stat_file.exists():
            raise RuntimeError(f"Incomplete submitted E5 eval: {split}")
        else:
            pred,metric=evaluate_frozen(selected_model,rows,"submission_"+split)
            save_json_once(pred_file,pred)
            save_json_once(stat_file,metric)
        measured_submission[split]=metric
    del selected_model
    if device=="cuda":torch.cuda.empty_cache()
print("SCORES FOR ACTUAL SUBMITTED CHECKPOINT:")
for k,v in measured_submission.items():
    print(f"  {k:13} {v['correct']:4}/{v['n']:<4}",flush=True)
print("No sealed Round3 final Test labels or predictions accessed.")

In [ ]:
# Step 8 — Compare historical E5, export dpo.pt and save provenance
ARCHIVED_E5=repo_root/"dpo/experiments/round9_random_multi_e3_e6/epoch_e05.pt"
replay_model,_=model_from_checkpoint(REPRODUCED_E5)
parity={"archived_e5_exists":ARCHIVED_E5.is_file()}
if ARCHIVED_E5.is_file():
    historical,_=model_from_checkpoint(ARCHIVED_E5)
    ref_weights=historical.state_dict()
    got_weights=replay_model.state_dict()
    if set(ref_weights)!=set(got_weights):
        raise RuntimeError("Replayed and archived E5 architectures/weight keys differ")
    max_abs_difference=max((got_weights[k]-ref_weights[k]).abs().max().item()
                           for k in ref_weights)
    exactly_equal=all(torch.equal(got_weights[k],ref_weights[k])
                      for k in ref_weights)
    parity.update({"archived_e5_sha256":sha256(ARCHIVED_E5),
                   "max_absolute_weight_delta":max_abs_difference,
                   "bitwise_weights_equal":exactly_equal})
    print("HISTORICAL E5 PARITY:",json.dumps(parity,indent=2),flush=True)
    if not exactly_equal:
        print("WARNING: Fresh run is NOT byte/weight-identical to archived E5. "
              "Do not cite historical E5 scores as the new run's score. "
              "The measured scores above are authoritative for this run.",flush=True)
    del historical
else:
    print("Historical E5 is absent. All stages trained from SFT; parity cannot be checked.")

# The archived E5 is the explicit historical selected model. A full replay
# run provides evidence and a numerical parity check; its weights are NOT
# silently substituted if a different GPU/PyTorch version produces drift.
if FINAL_EXPORT_POLICY == "archived_if_available" and ARCHIVED_E5.is_file():
    export_source=ARCHIVED_E5
    export_model,_=model_from_checkpoint(export_source)
    export_origin="selected_archived_E5"
else:
    export_source=REPRODUCED_E5
    export_model=replay_model
    export_origin="fully_retrained_notebook_E5"
print(f"SUBMISSION SOURCE {export_origin}: {export_source}",flush=True)

# Build the original course-compatible weights-only checkpoint.
clean_checkpoint={"model_args":sft_args,
                  "model_state_dict":{k:v.detach().cpu().clone()
                                      for k,v in export_model.state_dict().items()}}
del export_model,replay_model
if FAST_SMOKE:
    print("SMOKE PASS — refusing to publish a tiny-sample model as formal dpo.pt")
else:
    final_dir=repo_root/"dpo"
    formal=final_dir/"dpo.pt"
    artifact=RUN_DIR/"selected_e5_dpo.pt"
    if artifact.exists():
        a=torch.load(artifact,map_location="cpu",weights_only=True)
        if set(a["model_state_dict"])!=set(clean_checkpoint["model_state_dict"]):
            raise RuntimeError("Existing final E5 artifact keys changed")
        if any(not torch.equal(a["model_state_dict"][k],clean_checkpoint["model_state_dict"][k])
               for k in a["model_state_dict"]):
            raise RuntimeError("Existing final E5 artifact differs; refuse overwrite")
    else:
        atomic_save_torch(artifact,clean_checkpoint)
    # Preserve existing dpo/dpo.pt. The exported checkpoint origin is
    # explicit and never implicitly substituted.
    if formal.is_file() and sha256(formal)!=sha256(artifact):
        backup=RUN_DIR/"backups"/f"old_dpo_{sha256(formal)[:16]}.pt"
        backup.parent.mkdir(parents=True,exist_ok=True)
        if not backup.exists():shutil.copy2(formal,backup)
        print(f"BACKUP original dpo.pt -> {backup}",flush=True)
    if not formal.exists() or sha256(formal)!=sha256(artifact):
        temporary=formal.with_suffix(".pt.partial")
        if temporary.exists():raise RuntimeError(f"Stale final checkpoint file: {temporary}")
        shutil.copy2(artifact,temporary)
        temporary.replace(formal)
    if sha256(formal)!=sha256(artifact):
        raise RuntimeError("Final submission checkpoint copy SHA mismatch")
    print(f"FINAL ARTIFACT READY: {formal} SHA256={sha256(formal)}",flush=True)

report={
    "version":"sc3000_final_notebook_v1",
    "mode":"smoke" if FAST_SMOKE else "full_historical_replay",
    "selected":"Random E5 (third epoch of four-epoch E3–E6 schedule)",
    "sources":env_record, "generator_sources_and_manifests":generator_provenance,
    "source_random_100k_sha256":sha256(random_data_path),
    "trained_e5_epoch_checkpoint_sha256":sha256(REPRODUCED_E5),
    "export_checkpoint_origin":export_origin,
    "export_source_sha256":sha256(export_source),
    "archived_parity":parity,
    "measured_dev_scores_of_replay":measured,
    "measured_dev_scores_of_submitted_checkpoint":measured_submission,
    "historical_dev_reference_only":historical_e5,
    "stage_history":{
        key:{"checkpoint":str(file.relative_to(repo_root)),
             "sha256":sha256(file)}
        for key,file in stage_outputs.items()
    },
    "final_test_used":False,
    "limitations":["Seen-heavy teacher680","development-selected E5",
                   "CUDA nondeterminism possible","archived E5 parity disclosed"]
}
report_path=RUN_DIR/"final_reproduction_manifest.json"
if report_path.exists():
    old=read_json(report_path)
    # Timestamp/environment may differ between runs, but numerical outputs
    # and immutable source/selected checkpoint must remain consistent.
    if old["trained_e5_epoch_checkpoint_sha256"]!=report["trained_e5_epoch_checkpoint_sha256"]:
        raise RuntimeError("Existing submission manifest points at another E5")
else:
    save_json_once(report_path,report)
print(f"FINAL AUDIT PASS: {report_path}",flush=True)

In [ ]:
# Load the fine-tuned model
ckpt_path = REPRODUCED_E5 if FAST_SMOKE else dpo_dir / "dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=True)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        # Complete original Step 8 without changing test_set.
        completion,stop_reason=greedy_original_stop(gpt,prompt,MAX_NEW_TOKENS)
        print(prompt+completion)
        ###########################################################
